# NB6: Cost-Aware Dynamic Routing

**2-minute intro script:** This lab uses cost-aware routing as an architectural pattern, not a hosted service. You will write the route policy yourself in local Python: inspect a typed task summary, choose a model class, explain the choice, and record an auditable trace. No hosted routing endpoint is called, and the required lab needs no live model.


## The local routing path

```text
[Task text]
    -> [Typed TaskComplexity]
    -> [Your Python choose_model() policy]
    -> [Illustrative model registry: cost + latency]
    -> [RouteTraceRecord]
```

The task analyzer and sample cost table are deterministic fixtures. **You implement the decision logic in `choose_model()` yourself.** The code runs locally; no hosted hosted router is involved. The sample prices and latencies are teaching values, not provider quotes.


In [ ]:
from pydantic import BaseModel, ConfigDict
from typing import Literal
from enum import Enum
from datetime import datetime, timezone

class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")

class ModelClass(str, Enum):
    FAST_CHEAP = "fast-cheap"
    BALANCED = "balanced"
    REASONING = "reasoning"
    CODE_SPECIALIST = "code-spec"

class TaskComplexity(StrictModel):
    complexity: Literal["simple", "medium", "complex"]
    requires_reasoning: bool
    code_heavy: bool
    risk_level: Literal["low", "medium", "high"]

class ModelSpec(StrictModel):
    cost_label: str
    cost_usd: float
    latency_label: str
    latency_seconds: float

class RoutingDecision(StrictModel):
    model: ModelClass
    rationale: str
    estimated_cost: str
    estimated_latency: str
    estimated_cost_usd: float
    estimated_latency_seconds: float

class RouteTraceRecord(StrictModel):
    task: str
    analysis: TaskComplexity
    decision: RoutingDecision
    timestamp_utc: str

## Lab: implement the route policy in Python

Write `CostAwareRouter.choose_model()` yourself before running the demo below. It receives a validated `TaskComplexity` and must return `(ModelClass, rationale)`. Keep the decision logic in this local Python method; do not call a hosted routing service or route-selection endpoint.

Use this baseline precedence so safety and reasoning needs are not overridden by task size:

1. High risk -> `REASONING`.
2. Otherwise, if reasoning is required -> `REASONING`.
3. Otherwise, if code-heavy -> `CODE_SPECIALIST`.
4. Otherwise, if simple -> `FAST_CHEAP`.
5. Otherwise -> `BALANCED`.

Return a short rationale for every branch. Later, you can add a `SECURITY_SPECIALIST` class and update the first rule. The optional classifier cell below only returns task features; it never chooses the route for you.


In [ ]:
class CostAwareRouter:
    """Local Python routing exercise inspired by heterogeneous routing research.

    This is a transparent teaching adapter, not a learned orchestration system. It does
    not call a hosted hosted routing endpoint. Learners implement the
    model-selection policy in choose_model().
    """

    # Illustrative values for comparing policies; not current provider prices.
    MODEL_SPECS = {
        ModelClass.FAST_CHEAP: ModelSpec(
            cost_label="$0.01", cost_usd=0.01, latency_label="0.5s", latency_seconds=0.5
        ),
        ModelClass.BALANCED: ModelSpec(
            cost_label="$0.10", cost_usd=0.10, latency_label="2s", latency_seconds=2.0
        ),
        ModelClass.REASONING: ModelSpec(
            cost_label="$0.50", cost_usd=0.50, latency_label="10s", latency_seconds=10.0
        ),
        ModelClass.CODE_SPECIALIST: ModelSpec(
            cost_label="$0.05", cost_usd=0.05, latency_label="1s", latency_seconds=1.0
        ),
    }

    def __init__(self):
        self.route_trace: list[RouteTraceRecord] = []

    def analyze_task(self, task: str) -> TaskComplexity:
        # Deterministic local starter: extract features, make no network call.
        task_lower = task.lower()
        requires_reasoning = any(
            word in task_lower
            for word in ["why", "explain", "analyze", "design", "compare", "tradeoff"]
        )
        code_heavy = any(
            word in task_lower
            for word in ["code", "function", "implement", "debug", "jwt", "api", "refactor"]
        )

        if len(task.split()) < 10 and not requires_reasoning and not code_heavy:
            complexity = "simple"
        elif len(task.split()) > 18 or "architecture" in task_lower:
            complexity = "complex"
        else:
            complexity = "medium"

        if any(word in task_lower for word in ["security", "secure", "auth", "jwt", "password", "pii"]):
            risk_level = "high"
        elif "test" in task_lower or "demo" in task_lower:
            risk_level = "low"
        else:
            risk_level = "medium"

        return TaskComplexity(
            complexity=complexity,
            requires_reasoning=requires_reasoning,
            code_heavy=code_heavy,
            risk_level=risk_level,
        )

    def choose_model(self, analysis: TaskComplexity) -> tuple[ModelClass, str]:
        """Learner task: write and return the local routing policy."""
        # TODO: implement the priority rules from the lab instructions.
        # Return a pair: (ModelClass value, short human-readable rationale).
        raise NotImplementedError("Write the model-selection policy in Python")

    def route(self, task: str) -> RoutingDecision:
        analysis = self.analyze_task(task)
        model, rationale = self.choose_model(analysis)
        if model not in self.MODEL_SPECS:
            raise ValueError(f"No model specification registered for {model!r}")

        spec = self.MODEL_SPECS[model]
        decision = RoutingDecision(
            model=model,
            rationale=rationale,
            estimated_cost=spec.cost_label,
            estimated_latency=spec.latency_label,
            estimated_cost_usd=spec.cost_usd,
            estimated_latency_seconds=spec.latency_seconds,
        )
        self.route_trace.append(
            RouteTraceRecord(
                task=task,
                analysis=analysis,
                decision=decision,
                timestamp_utc=datetime.now(timezone.utc).isoformat(),
            )
        )
        return decision


In [ ]:
def demo_cost_aware_routing():
    router = CostAwareRouter()
    tasks = [
        "What's 2+2?",
        "Implement a secure JWT authentication system",
        "Write a function to sort a list",
        "Explain why microservices architecture is better than monolith",
        "Debug this race condition in concurrent code",
        "Create a demo landing page",
    ]

    for task in tasks:
        analysis = router.analyze_task(task)
        decision = router.route(task)
        print(f"Task: {task}")
        print(f"  Complexity: {analysis.complexity}")
        print(f"  Code-heavy: {analysis.code_heavy}")
        print(f"  Requires reasoning: {analysis.requires_reasoning}")
        print(f"  Risk: {analysis.risk_level}")
        print(f"  -> Route to: {decision.model.value}")
        print(f"     Rationale: {decision.rationale}")
        print(f"     Cost: {decision.estimated_cost}, Latency: {decision.estimated_latency}\n")

    print(f"Route trace records: {len(router.route_trace)}")
    print(router.route_trace[-1].model_dump_json(indent=2))

demo_cost_aware_routing()

In [ ]:
# Self-check: confirm the local policy respects risk, task type, and traceability.
router_check = CostAwareRouter()
route_cases = [
    ("What's 2+2?", ModelClass.FAST_CHEAP),
    ("Write a function to sort a list", ModelClass.CODE_SPECIALIST),
    ("Explain microservices tradeoffs", ModelClass.REASONING),
    ("Review payment workflow for PII", ModelClass.REASONING),
]

for task, expected_model in route_cases:
    decision = router_check.route(task)
    assert decision.model == expected_model, (
        f"{task!r}: expected {expected_model.value}, got {decision.model.value}"
    )
    assert decision.rationale.strip(), "Every route needs a rationale"

assert len(router_check.route_trace) == len(route_cases)
assert all(record.timestamp_utc.endswith("+00:00") for record in router_check.route_trace)
print("✅ Local routing policy passed the four checks and wrote UTC route traces.")


## Cost optimization challenge

cost-aware describes an architectural pattern; this exercise does not use a hosted routing API. Compare the decisions from your Python-written policy with a static baseline that sends every task to the reasoning class. The cost and latency values are illustrative so the trade-offs are deterministic and inspectable.


In [ ]:
def build_task_batch() -> list[str]:
    base_tasks = [
        "What's 2+2?",
        "Summarize this customer email",
        "Create a demo landing page",
        "Write a function to sort a list",
        "Debug this API timeout",
        "Implement secure JWT auth",
        "Explain microservices tradeoffs",
        "Analyze why checkout conversion dropped",
        "Design a multi-region architecture for payments",
        "Refactor this code path for readability",
    ]
    return [base_tasks[i % len(base_tasks)] for i in range(100)]

def compare_dynamic_vs_static(tasks: list[str]) -> dict:
    router = CostAwareRouter()
    dynamic_decisions = [router.route(task) for task in tasks]
    dynamic_cost = sum(decision.estimated_cost_usd for decision in dynamic_decisions)

    reasoning_spec = CostAwareRouter.MODEL_SPECS[ModelClass.REASONING]
    static_cost = len(tasks) * reasoning_spec.cost_usd

    savings = static_cost - dynamic_cost
    savings_percent = (savings / static_cost) * 100 if static_cost else 0.0

    return {
        "task_count": len(tasks),
        "dynamic_cost": round(dynamic_cost, 2),
        "static_reasoning_cost": round(static_cost, 2),
        "savings": round(savings, 2),
        "savings_percent": round(savings_percent, 1),
        "route_counts": {
            model.value: sum(1 for decision in dynamic_decisions if decision.model == model)
            for model in ModelClass
        },
    }

cost_report = compare_dynamic_vs_static(build_task_batch())
print(cost_report)
print(
    f"Dynamic routing saved {cost_report['savings_percent']}% "
    f"(${cost_report['savings']}) compared to static reasoning routing."
)

## Optional comparison: classifier output is not routing

This optional cell can use an OpenAI client to produce a typed `TaskComplexity`. It is separate from the required lab. The result does not select a model class: your local Python `choose_model()` remains the only routing policy. There is no hosted hosted routing endpoint.


In [ ]:
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    # Requires:
    #   pip install openai instructor
    #   export OPENAI_API_KEY="..."
    #
    # Optional comparison only: ask for TaskComplexity, not a route.
    # The learner-authored choose_model() policy remains local Python.
    # This is not a hosted routing endpoint.
    import os
    import instructor
    from openai import OpenAI

    client = instructor.from_openai(OpenAI(api_key=os.environ["OPENAI_API_KEY"]))

    live_analysis = client.chat.completions.create(
        model=os.environ.get("OPENAI_MODEL", "gpt-4o-mini"),
        response_model=TaskComplexity,
        messages=[
            {
                "role": "user",
                "content": "Classify this task for routing: Implement secure JWT auth",
            }
        ],
    )
    print(live_analysis.model_dump_json(indent=2))

## 🧪 Exercises: build an explainable local router

**Scenario:** A campaign has a fixed daily budget. Some tasks are simple, some need code expertise, and security-sensitive requests need deeper reasoning. The registry's costs are illustrative values for this exercise, not live prices.

**Your mission:**

1. Implement `CostAwareRouter.choose_model()` in Python using the precedence table above. Return both a model class and a clear rationale.
2. Add a `SECURITY_SPECIALIST` model class and registry entry. Route high-risk tasks to it before applying general cost rules.
3. Add a local budget counter. Before dispatch, reject a route that exceeds the remaining budget with a typed `BudgetExhausted` result.
4. Keep every decision in `route_trace` with a UTC timestamp, selected class, rationale, and estimated cost/latency.
5. Run the 100-task comparison and report the modeled savings against always choosing `REASONING`.

**Optional extension:** Use a schema-constrained classifier to produce `TaskComplexity`, then pass that typed result into your Python policy. The classifier does not choose the model class. No hosted routing service or hosted routing endpoint is used.

**Takeaway:** cost-aware is the architectural inspiration. In this lab, learners write and inspect a small, deterministic Python router; this is not a learned orchestration model.
